# Initial U-Net Tutorial / Work Through

Following the [WVView](https://wvview.org/dl/pytorch_examples/quarto/T13_UNet_semantic_segmentation.html) tutorial.

## 1. Create conda environment

`>>> conda env create -f unet-planetscope.yml`

## 2. Import torch package (pytorch) 

Also torch.nn subpackage and torchinfo package --- which lets us summarize the model

In [1]:
import torch 
import torch.nn as nn
from torchinfo import summary

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## 3. Define the U-Net

Defining multiple functions to help the implementation of a U-Net.

### 3a. Double Convolution Function

`double_conv()` performs the following process:

2D Convolution &rarr; D2 Batch Normalization &rarr; ReLU Activation &rarr; 2D Convolution &rarr; 2D Batch Normalization &rarr; ReLU Activation

<div style="border: 2px solid #000000; padding: 10px; border-radius: 5px;">
<strong>Batch Normalization:</strong> normalizes data within each mini-batch. Makes sure outputs of each layer stay steady as the model learns, helping the model train faster and learn more efficiently.

* <u>Faster Convergence:</u> reduces internal covariate shift, allowing for faster convergence during training
* <u>Higher Learning Rates:</u> allows for use of higher learning rates without the risk of divergence
* <u>Regularization Effect:</u> introduces a slight regularization effect that reduces the need for adding regularization techniques
</div>

<div style="border: 2px solid #000000; padding: 10px; border-radius: 5px;">
<strong>ReLU:</strong> Rectified Linear Unit, the most widely used activation function in deep learning. It outputs the input directly if it is positive and returns zero otherwise. 

$f(x) = max(0,x)$

* where $x$ is the input to the neuron
* $f(x)$ returns $x$ if $x > 0 $ and 0 if $f \leq 0 $
</div>

In [3]:
def double_conv(inChannels, outChannels):
  '''
    -------
    Params: 
    -------
      - inChannels: number of input channels or feature maps
      - outChannels: number of output channels or feature maps
    --------
    Returns:
    --------
      - nn.Sequential(): steps of our double convolution
  '''
  return nn.Sequential(
    # 1. 2D convolution, accepts defined number of input and output channels
    #    kernel size of 3x3 with a stride and padding of 1
    #    This will result in returning arrays with the same height and width as the input image or feature maps
      nn.Conv2d(inChannels, outChannels, kernel_size=(3,3), stride=1, padding=1),
    # 2. Normaluze data
      nn.BatchNorm2d(outChannels),
    # 3. Activation fucntion --- removes negatives
    #    inplace=True reduces memory consumption by applying the transformation in place
      nn.ReLU(inplace=True),
    # 4. 2D convolution, accepts the number of define output channels and returns the same number of augmented channels
      nn.Conv2d(outChannels, outChannels, kernel_size=(3,3), stride=1, padding=1),
    # 2. Normaluze data
      nn.BatchNorm2d(outChannels),
    # 3. Activation fucntion --- removes negatives
      nn.ReLU(inplace=True)
  )

### 3b. Decoder (upward convolution) Function

`up_conv()` performs the following process:

2d Transpose Convlolution &rarr; D2 Batch Normalization &rarr; ReLU Activation

<div style="border: 2px solid #000000; padding: 10px; border-radius: 5px;">
<strong>Transpose Convolution:</strong> 

* <u>Faster Convergence:</u> reduces internal covariate shift, allowing for faster convergence during training
* <u>Higher Learning Rates:</u> allows for use of higher learning rates without the risk of divergence
* <u>Regularization Effect:</u> introduces a slight regularization effect that reduces the need for adding regularization techniques
</div>